# config-grafana-dashboard — Static Grafana dashboards

Beyond mixin-rendered dashboards, the workspace can also pull released community dashboards as raw JSON — either from grafana.com (pinned by `id` + `revision`) or from any HTTP URL. Each release is reproducible (lock by SHA-256 if you want).

Sections:

1. List every `dashboards:` entry across `/config/*.yaml`.
2. Add a dashboard from grafana.com.
3. Add a dashboard from an arbitrary HTTP URL.
4. Sync, render, and inspect what was produced.

In [ ]:
CONFIG_DIR = '/config'
PICK_CONFIG = 'k8s-monitoring'   # one of the .yaml basenames in CONFIG_DIR

## 1. What's defined across all configs?

In [ ]:
import os, glob, yaml, pandas as pd

rows = []
for path in sorted(glob.glob(f'{CONFIG_DIR}/*.yaml')):
    cfg = yaml.safe_load(open(path)) or {}
    env = cfg.get('name', os.path.basename(path))
    for name, body in (cfg.get('dashboards') or {}).items():
        src = (body or {}).get('source', {})
        kind = next(iter(src.keys()), '')
        if kind == 'grafana':
            ref = f"grafana.com/{src['grafana']['id']} rev {src['grafana']['revision']}"
        elif kind == 'http':
            ref = src['http']['url']
        else:
            ref = ''
        cfg_block = body.get('config') or {}
        rows.append({
            'env': env,
            'dashboard': name,
            'source': kind,
            'pin': ref,
            'folder': cfg_block.get('grafanaDashboardFolder', ''),
            'uid': cfg_block.get('uid', ''),
        })

df = pd.DataFrame(rows)
print(f'{len(df)} dashboard entries across {df["env"].nunique() if not df.empty else 0} environments')
df

## 2. Add a grafana.com dashboard

Find the dashboard at `https://grafana.com/grafana/dashboards/<id>` and pin a specific `revision` (the release identifier — never use "latest"). Map its `${DS_*}` placeholders to your real datasource names.

```yaml
dashboards:
  node-exporter-full:
    source:
      grafana:
        id: 1860
        revision: 41          # required
    config:
      grafanaDashboardFolder: Platform
      uid: node-exporter-full
      datasources:
        DS_PROMETHEUS: prometheus
```

Quick sanity check — fetch the dashboard's `__inputs` so you know which `DS_*` placeholders to map:

In [ ]:
import requests

def grafana_com_inputs(dashboard_id, revision):
    url = f'https://grafana.com/api/dashboards/{dashboard_id}/revisions/{revision}/download'
    j = requests.get(url, timeout=30).json()
    return [(i.get('name'), i.get('pluginId') or i.get('type')) for i in j.get('__inputs', [])]

grafana_com_inputs(1860, 41)

## 3. Add an HTTP-sourced dashboard

Use this for self-hosted exports or vendor JSON not on grafana.com. The optional `sha256` pins the exact bytes.

```yaml
dashboards:
  loki-chunks:
    source:
      http:
        url: https://raw.githubusercontent.com/grafana/loki/main/production/loki-mixin-compiled/dashboards/loki-chunks.json
        # sha256: <hex digest>
    config:
      grafanaDashboardFolder: Logging
      uid: loki-chunks
```

To compute the SHA-256 to lock against:

In [ ]:
import hashlib, requests

url = 'https://raw.githubusercontent.com/grafana/loki/main/production/loki-mixin-compiled/dashboards/loki-chunks.json'
hashlib.sha256(requests.get(url, timeout=30).content).hexdigest()

## 4. Sync, render, and inspect output

In [ ]:
import subprocess

env_name = PICK_CONFIG
cfg_path = f'{CONFIG_DIR}/{env_name}.yaml'
src_dir = f'/source/{env_name}'
build_dir = f'/build/{env_name}'

shenv = os.environ | {'CONFIG_FILE': cfg_path, 'SOURCE_DIR': src_dir, 'BUILD_DIR': src_dir}
subprocess.run(['bash', '-lc', 'sync-dashboards'], env=shenv, check=False)
subprocess.run(['bash', '-lc', 'render-resources'],
               env=shenv | {'BUILD_DIR': build_dir}, check=False)

In [ ]:
import os
rows = []
fetched_dir = f'{src_dir}/dashboards'
if os.path.isdir(fetched_dir):
    for f in sorted(os.listdir(fetched_dir)):
        if f.endswith('.json'):
            rows.append({'fetched_dashboard': f, 'size_kb': round(os.path.getsize(os.path.join(fetched_dir, f)) / 1024, 1)})
pd.DataFrame(rows)

### Where to go next

- The static-dashboard pipeline reuses `apply-grizzly-grafana-dashboards` for delivery — same as mixin-rendered dashboards.
- Bumping a release: change `revision:` (grafana.com) or `sha256:` (http) and re-run sync.
- See [`config-mixin.ipynb`](config-mixin.ipynb) for jsonnet-rendered dashboards bundled with rules.